# Part 15 — Synthesis: atlas figures, municipal ranking, GEE App

Assemble the deliverables from the built assets: the thesis atlas figures (present + future), the municipal opportunity/vulnerability ranking over `municipal_godf`, and the interactive Earth Engine App. Figure rendering lives in `tools/make_figures.py`; the App source in `gee_js/atlas_app.js`.

**Output:** `docs/thesis/figures/*.png`, `docs/thesis/figures/municipal_ranking.csv`, and a published App URL (manual one-click). **DoD:** all RQs reported with uncertainty; atlas figures rendered; municipal rankings produced; App source ready to publish.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import ee, geemap
import utils, features
project = utils.init()
print('EE initialized; project =', project)
sys.path.insert(0, os.path.abspath('../tools'))
import external, make_figures

In [ ]:
aoi = utils.load_aoi(project)
print('AOI area (km^2):', round(aoi.area(1000).divide(1e6).getInfo(), 1))

### Render the atlas figures (present + future) → `docs/thesis/figures/`

In [ ]:
r = make_figures.Renderer()
for name in make_figures.PRESENT + make_figures.FUTURE:
    getattr(r, name)()

### Municipal opportunity / vulnerability ranking

In [ ]:
import pandas as pd
suit = ee.Image(utils.asset_id(project, 'suit_present'))
uu = ee.Image(utils.asset_id(project, 'realized_vs_potential'))
delta = ee.Image(utils.asset_id(project, 'delta_ssp585_2051_2070'))
muni = external.municipal_fc()
agg = (suit.select(['suit_soybean', 'suit_sugarcane'])
       .addBands(uu.select(['underused_soybean', 'underused_sugarcane']))
       .addBands(delta.select('delta_other_crops')))
cols = ['ADM2_NAME', 'suit_soybean', 'underused_soybean', 'delta_other_crops']
feats = external.municipal_means(muni, agg).select(cols, None, False).getInfo()['features']
df = pd.DataFrame([f['properties'] for f in feats]).dropna()
df['opportunity'] = df['suit_soybean'] * df['underused_soybean']
df.sort_values('opportunity', ascending=False).round(3).to_csv('figures/municipal_ranking.csv', index=False)
print('TOP OPPORTUNITY:'); print(df.nlargest(5, 'opportunity')[['ADM2_NAME','opportunity']].to_string(index=False))
print('TOP VULNERABILITY:'); print(df.nsmallest(5, 'delta_other_crops')[['ADM2_NAME','delta_other_crops']].to_string(index=False))

### Interactive GEE App
The App source is `gee_js/atlas_app.js` (layer selector across the seven `suit_*`, FAO classes, zones, realized use, under-utilization and the CMIP6 `delta_*`; click-to-read municipal values). **Publishing is a manual one-click step** — paste the script into the [Code Editor](https://code.earthengine.google.com), then **Apps ▸ NEW APP ▸ publish**. The Python API cannot deploy an App.